# Treino dos modelos do RNAmining por espécie

Este notebook existe para automatizar o treino de modelos do RNAmining a partir dos arquivos FASTA já preparados e organizados por espécie.

A preparação e a inspeção dos dados já foram feitas em outro notebook. Aqui, o foco é apenas utilizar os arquivos `cds` e `ncrna` de cada espécie para executar o fluxo de treino da ferramenta de forma organizada e reproduzível.

## O que ele faz

O notebook executa as seguintes etapas:

1. **Configurações de ambiente**  
   Imports e centralização dos caminhos para os FASTAs organizados, para o módulo de treino e para as saídas geradas durante a execução.

2. **Averiguar dados das espécies**  
   - Identifica, para cada espécie, os arquivos `cds` e `ncrna` disponíveis para treino. 
   - Confere se cada espécie possui os dois arquivos necessários para o treino.

3. **Execução automatizada do treino**

Para cada espécie, o notebook aciona o fluxo já implementado no módulo `train_model.py`.

- Geração dos arquivos `.arff`: Processa os FASTAs e gera os arquivos intermediários usados como entrada do modelo.
- Construção dos datasets: Lê os `.arff`, monta os datasets coding e noncoding e aplica a normalização já implementada no módulo da ferramenta.
- Balanceamento das classes: Ajusta os conjuntos coding e noncoding para que tenham o mesmo número de instâncias.
- Treino dos modelos: Treina um modelo XGBoost para cada espécie.
- Salvamento dos modelos e registro da execução: Salva os arquivos `.pkl` gerados e registra o status final do treino por espécie.

## Resultado esperado

Ao final, o notebook gera um modelo treinado por espécie, além de um registro da execução para facilitar conferência, rastreabilidade e uso posterior.

## 1. Configurações de ambiente

#### 1.1 Imports

In [1]:
from pathlib import Path
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import subprocess
import sys
import os

### 1.2 Paths

In [2]:
# Carrega o .env
load_dotenv()

# Pasta base do projeto RNAmining
BASE_RNAMINING = Path(os.environ.get("RNAMINING_DIR"))

# Pasta com os novos FASTAs do ensemble
S5_DIR = BASE_RNAMINING / "volumes" / "rnamining-front" / "data" / "S5_File"

# Pasta com os dados preparados (split + balanceamento)
TRAIN_TEST_SPLIT_DIR = S5_DIR / "Train_Test_Split"

# FASTAs codificantes
CODING_DIR = TRAIN_TEST_SPLIT_DIR / "coding"
# FASTAs não codificantes
NONCODING_DIR = TRAIN_TEST_SPLIT_DIR / "noncoding"

# Pasta onde estáo os scrpits do RNAmining
SCRIPTS_DIR = BASE_RNAMINING / "volumes" / "rnamining-front" / "assets" / "scripts"

# Pasta onde estáo os modulos
MODELS_DIR = BASE_RNAMINING / "volumes" / "rnamining-front" / "assets" / "scripts" / "models" / "coding_prediction"

In [3]:
# Adiciona essa pasta ao path do Python
sys.path.append(str(SCRIPTS_DIR))

# Importa as funções do módulo de treino
from model_train import process_inputfile, process_dataset, balance, xgboost_model

## 2. Execução automatizada do treino

Loop de treino por espécie

In [4]:
# Lista para registrar execução
rows = []

# Looping que percorre os arquivos coding_train
for coding_file in CODING_DIR.glob("*_coding_train.fa"):

    # Nome da espécie analisada
    species = coding_file.name.replace("_coding_train.fa", "")

    # Arquivo noncoding correspondente
    noncoding_file = NONCODING_DIR / f"{species}_noncoding_train.fa"

    # Se não existir o par, pula
    if not noncoding_file.exists():
        print(f"[SKIP] {species} sem noncoding_train")

        rows.append({
            "species": species,
            "status": "skip",
            "model_path": None
        })
        continue

    # Estou reutilizando o máximo possível do modulo "model_train.py"
    try:
        # Gera as ARFFs (features)
        coding_arff = process_inputfile(str(coding_file), f"{species}_coding")
        noncoding_arff = process_inputfile(str(noncoding_file), f"{species}_noncoding")

        # Constrói os datasets
        dataset_cod = process_dataset(coding_arff, cod=True)
        dataset_ncod = process_dataset(noncoding_arff, cod=False)

        # Balanceia os datasets
        final_dataset = balance(dataset_cod, dataset_ncod)

        # Separa X e y
        X_train = final_dataset.drop(columns=["cod"])
        y_train = final_dataset["cod"]

        # Caminho onde salvar os novos modelos
        model_output = MODELS_DIR / species

        # Treina e salva
        xgboost_model(X_train, y_train, str(model_output))

        print(f"[OK] {species}")

        rows.append({
            "species": species,
            "status": "ok",
            "model_path": str(model_output) + ".pkl"
        })

    except Exception as e:
        print(f"[ERROR] {species} -> {e}")

        rows.append({
            "species": species,
            "status": "error",
            "model_path": None
        })

[OK] Crocodylus_porosus
[OK] Gallus_gallus
[OK] Mus_musculus
[OK] Rattus_norvegicus
[OK] Ornithorhynchus_anatinus
[OK] Xenopus_tropicalis
[OK] Monodelphis_domestica
[OK] Anolis_carolinensis
[OK] Latimeria_chalumnae
[OK] Notechis_scutatus
[OK] Sphenodon_punctatus
[OK] Chrysemys_picta_bellii
[ERROR] Danio_rerio -> local variable 'first' referenced before assignment
[OK] Eptatretus_burgeri
[OK] Homo_sapiens
[OK] Petromyzon_marinus
